## Preprocesamiento y Modelado

Una vez inspeccionado el dataset en `customer_churn_eda.ipynb` definimos una una estrategia de preprocesamiento iterativo (de menos a más) para el encontrar

In [7]:
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import PowerTransformer, RobustScaler, StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
# Rutas de los archivos de datos
TRAIN_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/train.csv"
TEST_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"


# Cargamos de nueovo los datos para el pipeline
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
# Definir variables objetivo
TARGET = 'Exited'
# Variables numéricas: Incluyo las continuas y las binarias numéricas (HasCrCard, IsActiveMember)
NUM_FEATURES = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
# Variables categóricas: Las de texto con pocas categorías
CAT_FEATURES = ['Geography', 'Gender']
# Variables a eliminar inicialmente (IDs y apellido)
DROP_FEATURES = ['CustomerId', 'Surname']
RANDOM_STATE = 100            # Semilla para reproducibilidad

# 1. Preparación de Datos
# Separar X_train e y_train por convención
# X_train = variables independientes
# y_train = variable dependiente u objetivo
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

# Construcción de Pipelines de Preprocesamiento
# Dado que hay valores nulos en los datos,la imputación es obligatoria.

# Pipeline Numérico
num_pipeline = Pipeline(
    steps=[
    ('scaler', StandardScaler()), # KNN requiere escalar primero para calcular distancias bien.
    # 1. KNN Imputer: Rellena basándose en vecinos.  
    ('imputer', KNNImputer(n_neighbors=5, weights='uniform')),
    # Transformación: Yeo-Johnson intenta hacer la variable Gaussiana (normal)
    # Funciona mejor que logaritmo para datos con ceros y negativos.
    ('transformer', PowerTransformer(method='yeo-johnson')),
    # Escalado Robusto: Ignora outliers al escalar
    ('robust_scaler', RobustScaler())
])

# Pipeline Categórico
cat_pipeline = Pipeline(
    steps=[
    # Estrategia: Rellenar con el más frecuente (moda)
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # Codificación: OneHot para convertir "France", "Germany" en números (0, 1)
    # handle_unknown='ignore' es CRÍTICO para que no falle si en test aparece algo raro
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Preprocesador Maestro
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, NUM_FEATURES),
        ('cat', cat_pipeline, CAT_FEATURES)
    ],
    # 'remainder="drop"' elimina automáticamente CustomerId y Surname
    remainder='drop' 
)

# 3. Pipeline Completo (Preprocesamiento + Modelo)
# Empezamos con LinearDiscriminantAnalysis como línea base (simple y rápida) 
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
])

#4. Validación Local  Cruzada (Fase 4 adelantada)
# Configuración: 5 splits (divisiones) .
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Ejecutamos la validación.
# scoring='f1' métrica para Kaggle
cv_scores = cross_val_score(
    model_pipeline, 
    X_train, 
    y_train, 
    cv=cv_strategy, 
    scoring='f1',
    n_jobs=-1 # Nº de núcleos de CPU -1= todos 
)

# Resultados
print(f"--- Resultados de Validación Cruzada (5-Fold) ---")
print(f"F1-Scores individuales: {cv_scores}")
print(f"F1-Score Medio: {cv_scores.mean():.4f}")
print(f"Desviación Estándar: {cv_scores.std():.4f}")
# Interpretación de la desviación:
# Si es baja (< 0.02), tu modelo es estable.
# Si es alta, tu modelo depende de la suerte de los datos que le toquen.

# 5. Generación de Submission para Kaggle (Fase 5)
# Re-entrenamos con TODOS los datos de train para la predicción final
model_pipeline.fit(X_train, y_train) 
test_predictions = model_pipeline.predict(test_df)

# Crear fichero de salida
submission = pd.DataFrame({
    'CustomerId': test_df['CustomerId'],
    'Exited': test_predictions
})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Fichero '{SUBMISSION_PATH}' generado correctamente.")

--- Resultados de Validación Cruzada (5-Fold) ---
F1-Scores individuales: [0.34090909 0.38461538 0.30803571 0.35320088 0.36206897]
F1-Score Medio: 0.3498
Desviación Estándar: 0.0253
Fichero '/kaggle/working/submission.csv' generado correctamente.
